In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

In [ ]:
data = {

    "total_tasks": [
        20, 20, 20, 20, 20,
        15, 15, 15, 10, 10,
        25, 25, 18, 12, 8
    ],


    "completed_tasks": [
        18, 16, 14, 10, 5,
        13, 8, 6, 9, 3,
        24, 20, 15, 5, 2
    ],


    "attendance_rate": [
        95, 90, 85, 70, 50,
        92, 65, 55, 88, 45,
        98, 90, 80, 60, 40
    ],


    "activity_score": [
        95, 85, 80, 60, 30,
        90, 55, 40, 85, 25,
        98, 88, 75, 45, 20
    ],


    "progress_status": [
        "On Track",
        "On Track",
        "On Track",
        "Needs Attention",
        "Needs Attention",
        "On Track",
        "Needs Attention",
        "Needs Attention",
        "On Track",
        "Needs Attention",
        "On Track",
        "On Track",
        "On Track",
        "Needs Attention",
        "Needs Attention"
    ]
}


df = pd.DataFrame(data)


df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_model = df.copy()

encoder = LabelEncoder()

df_model["progress_status"] = encoder.fit_transform(
    df_model["progress_status"]
)

df_model.head()

In [ ]:
X = df_model[
    [
        "total_tasks",
        "completed_tasks",
        "attendance_rate",
        "activity_score"
    ]
]
y = df_model[
    "progress_status"
]



X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(
    X_train,
    y_train
)


print("Smart Progress AI Model trained successfully")

In [ ]:
y_pred = model.predict(
    X_test
)

accuracy = accuracy_score(
    y_test,
    y_pred
)


print(
    "Accuracy:",
    round(accuracy, 2)
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
joblib.dump(
    model,
    "smart_progress_model.pkl"
)

joblib.dump(
    encoder,
    "progress_encoder.pkl"
)


print("Smart Progress AI Model saved successfully")

In [ ]:
from fastapi import FastAPI
import joblib


app = FastAPI(
    title="Smart Progress Tracking AI API",
    description="Predict student progress status using Machine Learning",
    version="1.0"
)

model = joblib.load(
    "smart_progress_model.pkl"
)
encoder = joblib.load(
    "progress_encoder.pkl"
)


print("Smart Progress AI Model loaded successfully")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }



@app.post("/smart-progress")
def smart_progress(data: dict):

    input_data = [[

        data.get("total_tasks"),

        data.get("completed_tasks"),

        data.get("attendance_rate"),

        data.get("activity_score")

    ]]

    prediction = model.predict(
        input_data
    )[0]


    status = encoder.inverse_transform(
        [prediction]
    )[0]


    total = data.get("total_tasks", 0)

    completed = data.get("completed_tasks", 0)


    if total == 0:
        completion_rate = 0

    else:
        completion_rate = (
            completed / total
        ) * 100



    if status == "On Track":

        motivation = "Great progress! Keep going"

    else:

        motivation = "You need more focus and improvement "



    return {

        "studentId":
            data.get("studentId"),


        "completion_status":
            status,


        "completion_percentage":
            round(completion_rate, 2),


        "tasks_remaining":
            total - completed,


        "motivation":
            motivation
    }

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8001,
    log_level="info"
)

server = uvicorn.Server(config)
await server.serve()